# LPRNet Custom Training for Philippine License Plates

This notebook is designed to run on Google Colab to train a custom LPRNet (CRNN architecture with CTC loss) specifically for Philippine license plates.
It applies best practices from modern PyTorch training pipelines (`AdamW`, `bfloat16` mixed precision, `CosineAnnealingLR`, gradient clipping) while integrating Roboflow datasets and exporting directly to TensorFlow Lite for easy desktop deployment.

In [ ]:
!pip install roboflow ultralytics torch torchvision onnx onnx2tf tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Download your cropped plate dataset from Roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE") # PASTE API KEY
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1) # UPDATE VERSION

# Ensure you export as a standard Classification or Custom folder structure
dataset = version.download("folder")

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Philippine Plate Vocabulary (0-9, A-Z)
CHARS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
CHAR2LABEL = {char: i + 1 for i, char in enumerate(CHARS)}
LABEL2CHAR = {label: char for char, label in CHAR2LABEL.items()}
NUM_CLASSES = len(CHARS) + 1  # +1 for CTC Blank token (Index 0)

def preprocess_plate(image, target_size=(94, 24)):
    """ 
    Strictly mimics src/preprocess.py from the local project.
    Target Size: 94 width, 24 height
    """
    resized = cv2.resize(image, target_size, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    enhanced = clahe.apply(gray)
    blurred = cv2.GaussianBlur(enhanced, (3, 3), 0)
    sharpened = cv2.addWeighted(enhanced, 1.5, blurred, -0.5, 0)
    processed = cv2.cvtColor(sharpened, cv2.COLOR_GRAY2RGB)
    processed = processed.astype(np.float32) / 255.0
    
    # Convert to CHW format expected by PyTorch (Channels, Height, Width)
    return np.transpose(processed, (2, 0, 1))

class PlateDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.image_files = [f for f in os.listdir(img_dir) if f.endswith(".jpg") or f.endswith(".png")]
        
    def __len__(self):
        return len(self.image_files)
        
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Assume Roboflow exports images with filename formatting like ABC1234_jpg.rf... 
        # or if raw format: ABC1234.jpg
        label_str = img_name.split("_")[0].split(".")[0].upper()
        label = [CHAR2LABEL[c] for c in label_str if c in CHAR2LABEL]
        
        image = cv2.imread(img_path)
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
            
        tensor_img = preprocess_plate(image)
        return torch.tensor(tensor_img, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

def collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(images, 0)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    targets = torch.cat(labels)
    return images, targets, target_lengths

# UPDATE THIS PATH based on the dataset location structure
train_dataset = PlateDataset(f"{dataset.location}/train")
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)

In [ ]:
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, nclass, nh=256):
        super(CRNN, self).__init__()
        # CNN Backbone specifically tuned for 24x94 input size
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2), # 12 x 47
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2), # 6 x 23
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2, 2), (2, 1), (0, 1)), # 3 x 24
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2, 2), (2, 1), (0, 1)), # 1 x 25
            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True) # 1 x 24
        )
        # RNN for temporal sequences
        self.rnn = nn.LSTM(512, nh, bidirectional=True, batch_first=True)
        # Linear head for classification
        self.fc = nn.Linear(nh * 2, nclass)

    def forward(self, input):
        # input: [B, 3, 24, 94]
        conv = self.cnn(input)
        b, c, h, w = conv.size()
        conv = conv.squeeze(2) # [B, 512, W]
        conv = conv.permute(0, 2, 1) # [B, W, 512]
        
        rnn_out, _ = self.rnn(conv) # [B, W, nh*2]
        output = self.fc(rnn_out) # [B, W, nclass]
        
        # PyTorch CTC Loss requires [Sequence_Length, Batch, Num_Classes]
        return output.permute(1, 0, 2)

model = CRNN(nclass=NUM_CLASSES).cuda()
# ML Recipe: compile the model for dynamic fast execution on Ampere+ architectures like T4 (if supported) / A100
# (Using dynamic=False for max optimization)
model = torch.compile(model, dynamic=False)

In [ ]:
import math
import time

# ML Training Recipes:
# 1. AdamW Optimizer with beta1=0.9, beta2=0.95, eps=1e-10, weight_decay=0.1
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), eps=1e-10, weight_decay=0.1)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

# 2. Mixed Precision Setup for bf16
torch.set_float32_matmul_precision("high")
autocast_ctx = torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# 3. Cosine Annealing Learning Rate
def get_lr(step, total_steps, max_lr=1e-3, min_lr=1e-5, warmup_steps=50):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

epochs = 50
total_steps = len(train_loader) * epochs
step = 0

print("Starting Custom LPRNet Training...")
for epoch in range(epochs):
    model.train()
    for i, (images, targets, target_lengths) in enumerate(train_loader):
        images = images.cuda()
        targets = targets.cuda()
        
        lr = get_lr(step, total_steps)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
            
        with autocast_ctx:
            preds = model(images)
            # Sequence length depends on output W. For 94 width, W=24
            input_lengths = torch.full(size=(images.size(0),), fill_value=preds.size(0), dtype=torch.long)
            
            # CTC requires float32 for stable loss calculation
            loss = ctc_loss(preds.float(), targets, input_lengths, target_lengths)
            
        loss.backward()
        
        # 4. Gradient Clipping to prevent loss explosions
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
        optimizer.step()
        
        # 5. Efficient memory cleaning
        model.zero_grad(set_to_none=True) 
        
        if step % 20 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{step}/{total_steps}], Loss: {loss.item():.4f}, LR: {lr:.6f}")
            
        step += 1

# Save PyTorch Model native format to Drive
os.makedirs("/content/drive/MyDrive/parking_yolo", exist_ok=True)
torch.save(model.state_dict(), "/content/drive/MyDrive/parking_yolo/lprnet_ph_custom.pth")
print("Model saved to Drive!")

In [ ]:
# Convert PyTorch Model to ONNX
import onnx

model.eval()
# Provide dummy input using the 94x24 resolution
dummy_input = torch.randn(1, 3, 24, 94, device="cuda")

# We extract the raw module if it was wrapped by torch.compile
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model

torch.onnx.export(
    raw_model, 
    dummy_input, 
    "lprnet_ph.onnx", 
    export_params=True, 
    opset_version=13, 
    do_constant_folding=True, 
    input_names=["input"], 
    output_names=["output"], 
    dynamic_axes={"input": {0: "batch_size"}, "output": {1: "batch_size"}}
)

print("Exported to ONNX successfully.")

In [ ]:
# Convert ONNX to TensorFlow Lite for use in the local desktop application
!onnx2tf -i lprnet_ph.onnx -o /content/drive/MyDrive/parking_yolo/lprnet_ph_tflite
print("✅ Successfully exported to TFLite!")
print("Download the resulting .tflite file from your Google Drive and replace 'ph001.tflite' in the 'models' folder of your desktop project.")